### Actividad regresión logística

Dr. Hugo García Tecocoatzi

### Instrucciones:
Vel el ejemplo y realiza la actividad final.

In [ ]:
# Bibliotecas necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, accuracy_score)

import statsmodels.api as sm
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# Dataset: Breast Cancer Wisconsin (clasificación binaria: maligno vs benigno)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='Diagnostico')  # 0 = maligno, 1 = benigno

df = pd.concat([X, y], axis=1)

print(f"Forma del dataset: {df.shape}")
print(f"Clases: {dict(y.value_counts())}")
print(f"Proporción de benignos: {y.mean():.3f}")
df.head()

In [ ]:
# Para simplificar el ejercicio, usamos 4 variables representativas
vars_modelo = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']

print("Estadísticas descriptivas por clase:\n")
print(df.groupby('Diagnostico')[vars_modelo].agg(['mean', 'std']).round(3))

In [ ]:
# Boxplots por diagnóstico
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, var in enumerate(vars_modelo):
    sns.boxplot(x='Diagnostico', y=var, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{var} por Diagnóstico')
    axes[i].set_xlabel('Diagnóstico (0=maligno, 1=benigno)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre variables
plt.figure(figsize=(7, 6))
sns.heatmap(df[vars_modelo].corr(), annot=True, cmap='coolwarm',
            center=0, vmin=-1, vmax=1, fmt='.3f')
plt.title('Correlación entre variables predictoras')
plt.tight_layout()
plt.show()

In [ ]:
# Ajustar regresión logística con statsmodels (da p-valores y OR)
X_sm = sm.add_constant(df[vars_modelo])
y_sm = df['Diagnostico']

modelo_logit = sm.Logit(y_sm, X_sm).fit(disp=False)
print(modelo_logit.summary())

In [ ]:
# Odds Ratios con intervalos de confianza al 95%
odds_ratios = pd.DataFrame({
    'Coeficiente': modelo_logit.params,
    'OR': np.exp(modelo_logit.params),
    'IC 2.5%': np.exp(modelo_logit.conf_int()[0]),
    'IC 97.5%': np.exp(modelo_logit.conf_int()[1]),
    'p-valor': modelo_logit.pvalues
}).round(4)

print("=== ODDS RATIOS ===\n")
print(odds_ratios)

In [ ]:
# Predicciones de probabilidad
y_pred_prob = modelo_logit.predict(X_sm)
y_pred = (y_pred_prob > 0.5).astype(int)

# Matriz de confusión
print("Matriz de confusión:\n")
print(confusion_matrix(y_sm, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(y_sm, y_pred, target_names=['maligno', 'benigno']))

print(f"Accuracy: {accuracy_score(y_sm, y_pred):.4f}")

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_sm, y_pred_prob)
auc = roc_auc_score(y_sm, y_pred_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', lw=1, label='Aleatorio')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC - Regresión Logística')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Implementación de la prueba de Hosmer-Lemeshow
def hosmer_lemeshow(y_true, y_prob, g=10):
    """
    Prueba de bondad de ajuste de Hosmer-Lemeshow.
    y_true: valores observados (0/1)
    y_prob: probabilidades predichas
    g: número de grupos (por defecto 10)
    """
    # Crear DataFrame auxiliar
    df_hl = pd.DataFrame({'y': y_true, 'p': y_prob})
    df_hl['grupo'] = pd.qcut(df_hl['p'], q=g, duplicates='drop')
    
    # Calcular observados y esperados por grupo
    hl_tabla = df_hl.groupby('grupo').agg(
        n=('y', 'size'),
        observados=('y', 'sum'),
        esperados=('p', 'sum'),
        p_medio=('p', 'mean')
    )
    hl_tabla['obs_no'] = hl_tabla['n'] - hl_tabla['observados']
    hl_tabla['esp_no'] = hl_tabla['n'] - hl_tabla['esperados']
    
    # Estadístico HL
    HL = ((hl_tabla['observados'] - hl_tabla['esperados'])**2 /
          (hl_tabla['esperados'] * (1 - hl_tabla['p_medio']))).sum() \
       + ((hl_tabla['obs_no'] - hl_tabla['esp_no'])**2 /
          (hl_tabla['esp_no'] * (1 - hl_tabla['p_medio']))).sum()
    
    gl = len(hl_tabla) - 2
    p_valor = 1 - stats.chi2.cdf(HL, gl)
    
    return HL, gl, p_valor, hl_tabla

HL, gl, p_valor, tabla = hosmer_lemeshow(y_sm, y_pred_prob, g=10)

print(f"Estadístico HL: {HL:.4f}")
print(f"Grados de libertad: {gl}")
print(f"p-valor: {p_valor:.4f}\n")

if p_valor > 0.05:
    print("✅ No se rechaza H0: el modelo tiene buen ajuste")
else:
    print("⚠️ Se rechaza H0: posible falta de ajuste")

print("\nTabla por grupos:")
print(tabla.round(3))

In [ ]:
# Pseudo R²
ll_modelo = modelo_logit.llf        # log-verosimilitud del modelo completo
ll_nulo = modelo_logit.llnull       # log-verosimilitud del modelo nulo
n = len(y_sm)

# McFadden
r2_mcfadden = 1 - (ll_modelo / ll_nulo)

# Cox-Snell
r2_coxsnell = 1 - np.exp((2 / n) * (ll_nulo - ll_modelo))

# Nagelkerke (ajusta Cox-Snell para que el máximo sea 1)
r2_nagelkerke = r2_coxsnell / (1 - np.exp((2 / n) * ll_nulo))

print("=== PSEUDO R² ===\n")
print(f"Log-verosimilitud modelo completo: {ll_modelo:.4f}")
print(f"Log-verosimilitud modelo nulo:     {ll_nulo:.4f}\n")
print(f"McFadden:    {r2_mcfadden:.4f}")
print(f"Cox-Snell:   {r2_coxsnell:.4f}")
print(f"Nagelkerke:  {r2_nagelkerke:.4f}")

In [ ]:
# Comparación con sklearn
X_train, X_test, y_train, y_test = train_test_split(
    df[vars_modelo], df['Diagnostico'], test_size=0.3,
    random_state=42, stratify=df['Diagnostico']
)

# HL_w, gl_w, p_w, _ = hosmer_lemeshow(y_w_sm, y_prob_

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

modelo_sk = LogisticRegression(max_iter=1000)
modelo_sk.fit(X_train_sc, y_train)

y_pred_sk = modelo_sk.predict(X_test_sc)
y_prob_sk = modelo_sk.predict_proba(X_test_sc)[:, 1]

print("Accuracy en test: {:.4f}".format(accuracy_score(y_test, y_pred_sk)))
print("AUC en test:      {:.4f}".format(roc_auc_score(y_test, y_prob_sk)))
print("\nReporte de clasificación (test):\n")
print(classification_report(y_test, y_pred_sk, target_names=['maligno', 'benigno']))

In [16]:
# ACTIVIDAD: Replicar todo el análisis con el dataset de Vinos
# El dataset de vinos tiene 3 clases. Para hacer regresión logística binaria,
# debemos convertir el problema a dos clases. Por ejemplo:
#   - clase 0 (vino tinto A) vs. clases 1 y 2 (resto)
# O bien usar solo dos clases filtrando una.

print("ACTIVIDAD: Regresión Logística con el dataset de Vinos\n")
print("Instrucciones:")
print("1. Cargar el dataset con load_wine()")
print("2. Convertir el problema a binario (por ejemplo, clase 0 vs. resto)")
print("3. Seleccionar 3 o 4 variables predictoras")
print("4. Explorar los datos: estadísticas, boxplots, correlación")
print("5. Ajustar el modelo con statsmodels")
print("6. Interpretar los Odds Ratios")
print("7. Evaluar con matriz de confusión, ROC, Hosmer-Lemeshow y pseudo R²")
print("8. Comparar con sklearn")


ACTIVIDAD: Regresión Logística con el dataset de Vinos

Instrucciones:
1. Cargar el dataset con load_wine()
2. Convertir el problema a binario (por ejemplo, clase 0 vs. resto)
3. Seleccionar 3 o 4 variables predictoras
4. Explorar los datos: estadísticas, boxplots, correlación
5. Ajustar el modelo con statsmodels
6. Interpretar los Odds Ratios
7. Evaluar con matriz de confusión, ROC, Hosmer-Lemeshow y pseudo R²
8. Comparar con sklearn


In [ ]:
# ============================================================
# PLANTILLA PARA LA ACTIVIDAD CON EL DATASET DE VINOS
# Completa cada sección
# ============================================================

# 1. Cargar datos
# wine = load_wine()
# X_w = pd.DataFrame(wine.data, columns=wine.feature_names)
# y_w = pd.Series(wine.target, name='Clase')
# df_w = pd.concat([X_w, y_w], axis=1)


# 2. Convertir a binario
# df_w['Clase_bin'] = (df_w['Clase'] == 0).astype(int)


# 3. Seleccionar variables (elige 3 o 4)
# vars_w = ['alcohol', 'malic_acid', 'ash', 'total_phenols']

# 4. Visualización


# 5. Ajustar modelo con statsmodels
# X_w_sm = sm.add_constant(df_w[vars_w])
# y_w_sm = df_w['Clase_bin']
# modelo_w = sm.Logit(y_w_sm, X_w_sm).fit(disp=False)


# 6. Odds Ratios
# or_w = pd.DataFrame({
#     'Coef': modelo_w.params,
#     'OR': np.exp(modelo_w.params),
#     'IC 2.5%': np.exp(modelo_w.conf_int()[0]),
#     'IC 97.5%': np.exp(modelo_w.conf_int()[1]),
#     'p': modelo_w.pvalues
# }).round(4)

# 7. Evaluación
# y_prob_w = modelo_w.predict(X_w_sm)
# y_pred_w = (y_prob_w > 0.5).astype(int)


# HL_w, gl_w, p_w, _ = hosmer_lemeshow(y_w_sm, y_prob_w, g=10)

# ll_mod_w = modelo_w.llf
# ll_nul_w = modelo_w.llnull
# n_w = len(y_w_sm)
# r2_mcf_w = 1 - (ll_mod_w / ll_nul_w)
# print(f"McFadden R²: {r2_mcf_w:.4f}")

# 8. Comparar con sklearn